# NYC TLC — Table Profiler

Auto-profiles any table or view in the `taxi_portfolio` database.  
Column roles are inferred from two sources:
1. **`models/**/schema.yml`** — `unique`, `relationships`, `accepted_values: [0,1]` tests define PKs, FKs, and indicators
2. **`INFORMATION_SCHEMA.COLUMNS`** — data type used as fallback for everything else

Each column then gets type-appropriate analysis: distributions, null rates, cardinality, outliers, time-series charts.

**Setup:** credentials are read directly from `~/.dbt/profiles.yml` — the same file dbt uses to connect to Snowflake. No separate credential file needed.

In [1]:
import sys
print(sys.executable)

/Users/marcalexander/projects/evidence_project/.venv/bin/python


In [2]:
import os
import warnings
from pathlib import Path

import yaml
import snowflake.connector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (13, 4), 'axes.titlesize': 12})
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

# Project root = one level up from this notebook
PROJECT_ROOT = Path('..').resolve()
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/marcalexander/projects/ai_orchestrator_claude/portfolio_nyc_tlc


In [13]:
DBT_PROFILES = Path.home() / '.dbt' / 'profiles.yml'
DBT_PROFILE_NAME = 'nyc_taxi'   # must match the top-level key in profiles.yml


def load_profile(profiles_path: Path, profile_name: str) -> dict:
    """Parse ~/.dbt/profiles.yml and return the active output's connection dict."""
    if not profiles_path.exists():
        raise FileNotFoundError(
            f'profiles.yml not found at {profiles_path}\n'
            'Run  dbt init  or create ~/.dbt/profiles.yml manually.'
        )
    with open(profiles_path) as f:
        profiles = yaml.safe_load(f)

    if profile_name not in profiles:
        available = list(profiles.keys())
        raise KeyError(
            f'Profile "{profile_name}" not found in {profiles_path}.\n'
            f'Available profiles: {available}'
        )

    profile = profiles[profile_name]
    target  = profile.get('target', 'dev')
    outputs = profile.get('outputs', {})

    if target not in outputs:
        raise KeyError(
            f'Target "{target}" not found under profile "{profile_name}".\n'
            f'Available targets: {list(outputs.keys())}'
        )

    creds = outputs[target]
    if creds.get('type') != 'snowflake':
        raise ValueError(
            f'Expected type=snowflake, got type={creds.get("type")}.'
        )
    return creds


def get_connection():
    from cryptography.hazmat.primitives.serialization import (
        load_pem_private_key, Encoding, PrivateFormat, NoEncryption
    )
    creds = load_profile(DBT_PROFILES, DBT_PROFILE_NAME)

    connect_kwargs = dict(
        account   = creds['account'],
        user      = creds['user'],
        warehouse = creds.get('warehouse'),
        database  = creds.get('database'),
        schema    = creds.get('schema'),
    )
    # Only pass role if explicitly set in profiles.yml — never assume a default
    if creds.get('role'):
        connect_kwargs['role'] = creds['role']

    if 'private_key_path' in creds:
        key_path   = Path(creds['private_key_path']).expanduser()
        passphrase = creds.get('private_key_passphrase')
        passphrase = passphrase.encode() if passphrase else None
        with open(key_path, 'rb') as f:
            private_key = load_pem_private_key(f.read(), password=passphrase)
        connect_kwargs['private_key'] = private_key.private_bytes(
            encoding=Encoding.DER,
            format=PrivateFormat.PKCS8,
            encryption_algorithm=NoEncryption(),
        )
    else:
        connect_kwargs['password'] = creds['password']

    return snowflake.connector.connect(**connect_kwargs)


try:
    conn = get_connection()
    creds_preview = load_profile(DBT_PROFILES, DBT_PROFILE_NAME)
    print('Connected to Snowflake')
    print(f'  account   : {creds_preview["account"]}')
    print(f'  user      : {creds_preview["user"]}')
    print(f'  warehouse : {creds_preview.get("warehouse")}')
    print(f'  database  : {creds_preview.get("database")}')
    print(f'  schema    : {creds_preview.get("schema")}')
    print(f'  role      : {creds_preview.get("role", "(not set — using account default)")}')
except Exception as e:
    print(f'Connection failed: {e}')

Connected to Snowflake
  account   : dkphlsq-dwb04623
  user      : MARC4DATA
  warehouse : transforming
  database  : analytics
  schema    : dbt_malex
  role      : (not set — using account default)


In [14]:
# ── Schema YAML parser ────────────────────────────────────────────────────────
# Reads all models/**/schema.yml files and extracts column roles from dbt tests.
#
# Mapping logic:
#   unique test                         → surrogate_key (if col ends in _id) or primary_key
#   relationships test                  → foreign_key
#   accepted_values with values [0,1]   → indicator

def load_schema_roles(project_root: Path) -> dict:
    """
    Returns: {MODEL_NAME: {COLUMN_NAME: {yaml_role, tests, description}}}
    All keys are uppercased for case-insensitive lookup.
    """
    schema_roles = {}

    for schema_file in project_root.glob('models/**/schema.yml'):
        with open(schema_file) as f:
            doc = yaml.safe_load(f)

        for model in doc.get('models', []):
            model_name = model['name'].upper()
            col_map = {}

            for col in model.get('columns', []):
                col_name = col['name'].upper()
                tests    = col.get('tests', [])

                has_unique        = False
                has_relationships = False
                has_indicator     = False
                test_names        = []

                for t in tests:
                    if isinstance(t, str):
                        test_names.append(t)
                        if t == 'unique':
                            has_unique = True
                    elif isinstance(t, dict):
                        key = list(t.keys())[0]
                        test_names.append(key)
                        if key == 'relationships':
                            has_relationships = True
                        if key == 'accepted_values':
                            vals = t['accepted_values'].get('values', [])
                            if set(vals) == {0, 1}:
                                has_indicator = True

                if has_unique and col_name.endswith('_ID'):
                    yaml_role = 'surrogate_key'
                elif has_unique:
                    yaml_role = 'primary_key'
                elif has_relationships:
                    yaml_role = 'foreign_key'
                elif has_indicator:
                    yaml_role = 'indicator'
                else:
                    yaml_role = None

                col_map[col_name] = {
                    'yaml_role':   yaml_role,
                    'tests':       test_names,
                    'description': col.get('description', ''),
                }

            schema_roles[model_name] = col_map

    print(f'Loaded schema roles for {len(schema_roles)} models:')
    for m in sorted(schema_roles):
        print(f'  {m}  ({len(schema_roles[m])} columns with tests)')
    return schema_roles


schema_roles = load_schema_roles(PROJECT_ROOT)

Loaded schema roles for 9 models:
  DIM_DATE  (1 columns with tests)
  DIM_ZONES  (3 columns with tests)
  FCT_DAILY_DEMAND  (4 columns with tests)
  FCT_TRIPS  (10 columns with tests)
  INT_DAILY_DEMAND  (3 columns with tests)
  INT_TRIPS_ENRICHED  (3 columns with tests)
  STG_TAXI_ZONES  (2 columns with tests)
  STG_WEATHER  (3 columns with tests)
  STG_YELLOW_TRIPS  (13 columns with tests)


In [15]:
# ── Snowflake helpers ─────────────────────────────────────────────────────────

def sql(query: str) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    cur = conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    cols = [d[0].lower() for d in cur.description]
    return pd.DataFrame(rows, columns=cols)


def get_available_tables() -> pd.DataFrame:
    return sql("""
        SELECT table_schema, table_name, table_type,
               COALESCE(row_count, 0) AS row_count
        FROM   information_schema.tables
        WHERE  table_schema NOT IN ('INFORMATION_SCHEMA')
        ORDER  BY table_schema, table_name
    """)


def get_column_metadata(schema: str, table: str) -> pd.DataFrame:
    return sql(f"""
        SELECT column_name, data_type, is_nullable,
               character_maximum_length,
               numeric_precision, numeric_scale,
               ordinal_position
        FROM   information_schema.columns
        WHERE  table_schema = '{schema}'
          AND  table_name   = '{table}'
        ORDER  BY ordinal_position
    """)


print('Helpers ready.')

Helpers ready.


In [16]:
# ── Role inference ────────────────────────────────────────────────────────────
# Schema YAML takes precedence; dtype-based heuristics are the fallback.

ROLE_COLORS = {
    'surrogate_key': '#5B9BD5',
    'primary_key':   '#5B9BD5',
    'foreign_key':   '#ED7D31',
    'indicator':     '#70AD47',
    'label':         '#FFC000',
    'datetime':      '#9B59B6',
    'measure':       '#E74C3C',
    'categorical':   '#2ECC71',
    'other':         '#95A5A6',
}


def dtype_role(col_name: str, data_type: str) -> str:
    """Fallback role inference from column name pattern + Snowflake data type."""
    c  = col_name.lower()
    dt = data_type.upper()

    if c.endswith('_id') and any(t in dt for t in ('TEXT', 'VARCHAR', 'STRING')):
        return 'surrogate_key'
    if c.endswith('_id') and any(t in dt for t in ('NUMBER', 'INT', 'FIXED')):
        return 'foreign_key'
    if c.endswith('_ind'):
        return 'indicator'
    if c.endswith(('_label', '_name', '_flag', '_type')):
        return 'label'
    if any(t in dt for t in ('TIMESTAMP', 'DATE', 'TIME')):
        return 'datetime'
    if any(t in dt for t in ('FLOAT', 'REAL', 'DOUBLE')) or (
       any(t in dt for t in ('NUMBER', 'FIXED', 'NUMERIC')) and not c.endswith('_id')):
        return 'measure'
    if any(t in dt for t in ('TEXT', 'VARCHAR', 'STRING', 'CHAR')):
        return 'categorical'
    return 'other'


def resolve_role(table_name: str, col_name: str, data_type: str) -> str:
    """YAML role wins; dtype_role is the fallback."""
    yaml_col  = schema_roles.get(table_name.upper(), {}).get(col_name.upper(), {})
    yaml_role = yaml_col.get('yaml_role')
    return yaml_role if yaml_role else dtype_role(col_name, data_type)


print('Role inference ready.')

Role inference ready.


In [17]:
# ── Per-role profiling functions ──────────────────────────────────────────────

def _header(col, role, data_type, nullable, description=''):
    color = ROLE_COLORS.get(role, '#95A5A6')
    desc_html = f'<span style="color:#888; font-size:0.82em; margin-left:14px">{description}</span>' if description else ''
    display(HTML(f"""
    <div style="margin:16px 0 4px; padding:6px 14px;
                background:{color}18; border-left:4px solid {color}; border-radius:3px;">
      <strong style="font-size:1.05em">{col}</strong>
      <span style="margin-left:10px; color:#666; font-size:0.82em">{data_type}</span>
      <span style="margin-left:8px; background:{color}; color:#fff; font-size:0.72em;
                   padding:1px 7px; border-radius:10px">{role}</span>
      {'<span style="margin-left:8px; color:#bbb; font-size:0.78em">nullable</span>' if nullable == 'YES' else ''}
      {desc_html}
    </div>"""))


def _fmt_pct(n, total):
    return f'{n:,}  ({n / total * 100:.2f}%)' if total else 'N/A'


# ── Surrogate / primary key ───────────────────────────────────────────────────
def profile_key(fqn, col, n_total):
    r = sql(f"""
        SELECT COUNT(*) total,
               COUNT({col}) non_null,
               COUNT(DISTINCT {col}) distinct_vals
        FROM {fqn}""").iloc[0]
    null_n   = n_total - int(r['non_null'])
    dups     = int(r['non_null']) - int(r['distinct_vals'])
    uniq_pct = int(r['distinct_vals']) / int(r['non_null']) * 100 if r['non_null'] else 0
    status   = '100% unique' if dups == 0 else f'{uniq_pct:.2f}% unique  ({dups:,} duplicates)'
    display(pd.DataFrame([{
        'Total Rows':     f"{n_total:,}",
        'Null Count':     _fmt_pct(null_n, n_total),
        'Distinct':       f"{int(r['distinct_vals']):,}",
        'Uniqueness':     status,
    }]))


# ── Foreign key ───────────────────────────────────────────────────────────────
def profile_foreign_key(fqn, col, n_total):
    r = sql(f"""
        SELECT COUNT({col}) non_null,
               COUNT(DISTINCT {col}) distinct_vals,
               MIN({col}) min_val, MAX({col}) max_val
        FROM {fqn}""").iloc[0]
    null_n = n_total - int(r['non_null'])
    display(pd.DataFrame([{
        'Non-Null':  f"{int(r['non_null']):,}",
        'Null':      _fmt_pct(null_n, n_total),
        'Distinct':  f"{int(r['distinct_vals']):,}",
        'Min':       r['min_val'],
        'Max':       r['max_val'],
    }]))


# ── Indicator (0/1) ───────────────────────────────────────────────────────────
def profile_indicator(fqn, col, n_total):
    r = sql(f"""
        SELECT {col} AS val, COUNT(*) AS cnt
        FROM {fqn}
        GROUP BY {col} ORDER BY val NULLS LAST""").copy()
    r['pct'] = r['cnt'] / n_total * 100
    display(r.assign(
        cnt = r['cnt'].map('{:,}'.format),
        pct = r['pct'].map('{:.2f}%'.format)
    ))
    clean = r.dropna(subset=['val'])
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(clean['val'].astype(str), clean['cnt'].str.replace(',', '', regex=False).astype(int),
           color=ROLE_COLORS['indicator'], width=0.5)
    ax.set_title(col); ax.set_xlabel('Value'); ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout(); plt.show()


# ── Label / categorical (low-cardinality string) ──────────────────────────────
def profile_label(fqn, col, n_total):
    r = sql(f"""
        SELECT {col} AS val, COUNT(*) AS cnt
        FROM {fqn}
        GROUP BY {col} ORDER BY cnt DESC LIMIT 30""")
    null_n  = r.loc[r['val'].isna(), 'cnt'].sum()
    r       = r.dropna(subset=['val'])
    r['pct'] = r['cnt'] / n_total * 100
    display(HTML(f'<small>Distinct: {len(r):,} &nbsp;|&nbsp; Nulls: {_fmt_pct(int(null_n), n_total)}</small>'))
    display(r[['val', 'cnt', 'pct']].rename(columns={'val': col, 'cnt': 'count', 'pct': '%'})
             .assign(count=r['cnt'].map('{:,}'.format), **{'%': r['pct'].map('{:.2f}%'.format)}))


# ── Datetime ──────────────────────────────────────────────────────────────────
def profile_datetime(fqn, col, n_total):
    r = sql(f"""
        SELECT COUNT({col}) non_null,
               MIN({col})   min_val,
               MAX({col})   max_val
        FROM {fqn}""").iloc[0]
    null_n = n_total - int(r['non_null'])
    try:
        span = (pd.to_datetime(str(r['max_val'])) - pd.to_datetime(str(r['min_val']))).days
    except Exception:
        span = None
    display(pd.DataFrame([{
        'Non-Null':   f"{int(r['non_null']):,}",
        'Null':       _fmt_pct(null_n, n_total),
        'Min':        r['min_val'],
        'Max':        r['max_val'],
        'Span (days)':f"{span:,}" if span is not None else 'N/A',
    }]))
    # Monthly distribution
    dist = sql(f"""
        SELECT DATE_TRUNC('month', CAST({col} AS DATE)) AS month, COUNT(*) cnt
        FROM {fqn}
        WHERE {col} IS NOT NULL
        GROUP BY 1 ORDER BY 1""")
    if len(dist) > 1:
        fig, ax = plt.subplots(figsize=(13, 3))
        ax.bar(range(len(dist)), dist['cnt'], color=ROLE_COLORS['datetime'], width=0.8)
        ax.set_xticks(range(len(dist)))
        ax.set_xticklabels(dist['month'].astype(str), rotation=45, ha='right', fontsize=8)
        ax.set_title(f'{col} — monthly distribution')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        plt.tight_layout(); plt.show()


# ── Numeric measure ───────────────────────────────────────────────────────────
def profile_measure(fqn, col, n_total):
    r = sql(f"""
        SELECT COUNT({col})  AS non_null,
               SUM(CASE WHEN {col} = 0 THEN 1 ELSE 0 END)  AS zero_count,
               SUM(CASE WHEN {col} < 0 THEN 1 ELSE 0 END)  AS neg_count,
               MIN({col})   AS min_val,
               MAX({col})   AS max_val,
               AVG({col})   AS mean_val,
               MEDIAN({col}) AS median_val,
               STDDEV({col}) AS std_val,
               PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY {col}) AS p01,
               PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY {col}) AS p25,
               PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY {col}) AS p75,
               PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY {col}) AS p95,
               PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY {col}) AS p99
        FROM {fqn}""").iloc[0]

    null_n = n_total - int(r['non_null'])

    def _f(v):
        return f'{float(v):,.4f}' if v is not None else 'N/A'

    stats = pd.DataFrame([{
        'Count':     f"{int(r['non_null']):,}",
        'Nulls':     _fmt_pct(null_n, n_total),
        'Zeros':     _fmt_pct(int(r['zero_count']), n_total),
        'Negatives': _fmt_pct(int(r['neg_count']), n_total),
        'Min':       _f(r['min_val']),
        'P01':       _f(r['p01']),
        'P25':       _f(r['p25']),
        'Median':    _f(r['median_val']),
        'Mean':      _f(r['mean_val']),
        'P75':       _f(r['p75']),
        'P95':       _f(r['p95']),
        'P99':       _f(r['p99']),
        'Max':       _f(r['max_val']),
        'Std Dev':   _f(r['std_val']),
    }])
    display(stats.T.rename(columns={0: 'Value'}))

    # Histogram bucketed in Snowflake (P01–P99 clips outliers)
    try:
        p01, p99 = float(r['p01']), float(r['p99'])
        if p01 < p99:
            hist = sql(f"""
                SELECT WIDTH_BUCKET({col}, {p01}, {p99}, 30) AS bucket,
                       MIN({col}) AS bucket_floor,
                       COUNT(*)   AS cnt
                FROM {fqn}
                WHERE {col} IS NOT NULL
                  AND {col} BETWEEN {p01} AND {p99}
                GROUP BY 1 ORDER BY 1""")
            if len(hist) > 1:
                fig, ax = plt.subplots(figsize=(13, 3))
                width = (p99 - p01) / 30 * 0.9
                ax.bar(hist['bucket_floor'].astype(float), hist['cnt'],
                       width=width, color=ROLE_COLORS['measure'], align='edge')
                ax.set_title(f'{col} — distribution  (P01 – P99, outliers clipped)')
                ax.set_xlabel(col)
                ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
                plt.tight_layout(); plt.show()
    except Exception:
        pass


# ── High-cardinality categorical ──────────────────────────────────────────────
def profile_categorical(fqn, col, n_total):
    r = sql(f"""
        SELECT COUNT({col})          AS non_null,
               COUNT(DISTINCT {col}) AS distinct_vals,
               AVG(LEN({col}))       AS avg_len,
               MAX(LEN({col}))       AS max_len
        FROM {fqn}""").iloc[0]
    null_n = n_total - int(r['non_null'])
    display(pd.DataFrame([{
        'Non-Null':    f"{int(r['non_null']):,}",
        'Null':        _fmt_pct(null_n, n_total),
        'Distinct':    f"{int(r['distinct_vals']):,}",
        'Avg Length':  f"{float(r['avg_len']):.1f}" if r['avg_len'] else 'N/A',
        'Max Length':  r['max_len'],
    }]))
    # Top values (always shown; chart if low cardinality)
    top = sql(f"""
        SELECT {col} AS val, COUNT(*) AS cnt
        FROM {fqn}
        GROUP BY {col} ORDER BY cnt DESC LIMIT 20""")
    top = top.dropna(subset=['val'])
    top['pct'] = top['cnt'] / n_total * 100
    if int(r['distinct_vals']) <= 30:
        fig, ax = plt.subplots(figsize=(9, max(3, len(top) * 0.35)))
        ax.barh(top['val'].astype(str)[::-1], top['cnt'].values[::-1],
                color=ROLE_COLORS['categorical'])
        ax.set_title(f'{col} — value counts')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        plt.tight_layout(); plt.show()
    else:
        display(top.assign(cnt=top['cnt'].map('{:,}'.format),
                           pct=top['pct'].map('{:.2f}%'.format))
                   .rename(columns={'val': col, 'cnt': 'count', 'pct': '%'}))


print('Profilers ready.')

Profilers ready.


In [18]:
# ── Main orchestrator ─────────────────────────────────────────────────────────

PROFILERS = {
    'surrogate_key': profile_key,
    'primary_key':   profile_key,
    'foreign_key':   profile_foreign_key,
    'indicator':     profile_indicator,
    'label':         profile_label,
    'datetime':      profile_datetime,
    'measure':       profile_measure,
    'categorical':   profile_categorical,
}


def profile_table(schema_table: str, sample_pct: float = 10):
    """
    schema_table : 'SCHEMA.TABLE_NAME'  e.g. 'DBT_MALEX_MARTS.FCT_TRIPS'
    sample_pct   : Bernoulli sample percentage (0 = full table, 10 = 10%, etc.)
                   Bernoulli sampling selects each row independently at random,
                   so the sample is representative across the full dataset.
    """
    schema, table = schema_table.upper().split('.')

    # SAMPLE BERNOULLI selects each row with independent probability — truly random.
    # Key uniqueness checks always run on the full table regardless of sampling.
    sample_clause = f'SAMPLE BERNOULLI ({sample_pct})' if sample_pct > 0 else ''
    fqn_sample    = f'{schema_table} {sample_clause}'.strip()
    fqn_full      = schema_table

    n_total = int(sql(f'SELECT COUNT(*) n FROM {fqn_sample}').iloc[0]['n'])

    sampled_note = f'  (Bernoulli {sample_pct}% sample)' if sample_pct else ''
    display(HTML(f"""
    <h2 style="margin-bottom:2px">{schema_table}</h2>
    <p style="color:#666; margin-top:0; font-size:0.9em">
      {n_total:,} rows{sampled_note}
    </p>"""))

    # Column metadata
    cols = get_column_metadata(schema, table)
    yaml_model = schema_roles.get(table, {})

    cols['role']        = cols.apply(
        lambda r: resolve_role(table, r['column_name'], r['data_type']), axis=1)
    cols['description'] = cols['column_name'].str.upper().map(
        lambda c: yaml_model.get(c, {}).get('description', ''))
    cols['yaml_tests']  = cols['column_name'].str.upper().map(
        lambda c: ', '.join(yaml_model.get(c, {}).get('tests', [])))

    # ── Column summary table ──────────────────────────────────────────────────
    display(HTML('<h3 style="margin-top:16px">Column Summary</h3>'))
    summary = cols[['column_name', 'data_type', 'role', 'is_nullable', 'yaml_tests']].copy()
    summary.columns = ['Column', 'Type', 'Role', 'Nullable', 'dbt Tests']

    def _color_role(val):
        c = ROLE_COLORS.get(val, '#95A5A6')
        return f'background-color: {c}22; color: #333'

    display(summary.style.applymap(_color_role, subset=['Role']))

    # ── Per-column profiles ───────────────────────────────────────────────────
    display(HTML('<hr><h3>Column Profiles</h3>'))

    for _, row in cols.iterrows():
        col   = row['column_name']
        role  = row['role']
        desc  = row['description'] or ''
        _header(col, role, row['data_type'], row['is_nullable'], desc)

        profiler = PROFILERS.get(role)
        if profiler:
            try:
                target = fqn_full if role in ('surrogate_key', 'primary_key') else fqn_sample
                profiler(target, col, n_total)
            except Exception as e:
                display(HTML(f'<p style="color:#c0392b; font-size:0.85em">Error: {e}</p>'))
        else:
            display(HTML('<p style="color:#999; font-size:0.85em">No profiler for this role.</p>'))


print('profile_table() ready.')

profile_table() ready.


In [ ]:
# ── PDF export ────────────────────────────────────────────────────────────────
import io, base64
from IPython.utils.capture import capture_output

def profile_and_save(schema_table: str, sample_pct: float = 10):
    """
    Runs profile_table() and saves the full output to a PDF.
    File written to:  <project_root>/output/<profile>-<schema>-<table>.pdf
    """
    import weasyprint

    output_dir = PROJECT_ROOT / 'output'
    output_dir.mkdir(exist_ok=True)

    schema, table = schema_table.lower().split('.')
    pdf_path = output_dir / f'{DBT_PROFILE_NAME}-{schema}-{table}.pdf'

    # Capture everything profile_table() displays
    with capture_output() as cap:
        profile_table(schema_table, sample_pct)

    # Reassemble captured outputs into a single HTML document
    html_parts = ["""
    <style>
      body  { font-family: Arial, sans-serif; font-size: 10pt; margin: 24px; color: #222; }
      h2    { font-size: 14pt; margin-bottom: 2px; }
      h3    { font-size: 11pt; margin: 14px 0 4px; }
      hr    { border: none; border-top: 1px solid #ddd; margin: 10px 0; }
      table { border-collapse: collapse; width: 100%; margin: 6px 0; font-size: 8.5pt; }
      th, td{ border: 1px solid #ccc; padding: 3px 7px; }
      th    { background: #f2f2f2; font-weight: bold; }
      img   { max-width: 100%; margin: 6px 0; }
      pre   { font-size: 8pt; background: #f9f9f9; padding: 6px; border: 1px solid #eee; }
      div   { page-break-inside: avoid; }
    </style>
    """]

    for output in cap.outputs:
        if not hasattr(output, 'data'):
            continue
        data = output.data
        if 'text/html' in data:
            html_parts.append(data['text/html'])
        elif 'image/png' in data:
            raw = data['image/png']
            b64 = raw if isinstance(raw, str) else base64.b64encode(raw).decode()
            html_parts.append(f'<img src="data:image/png;base64,{b64}">')
        elif 'text/plain' in data:
            html_parts.append(f'<pre>{data["text/plain"]}</pre>')

    full_html = (
        '<!DOCTYPE html><html><head><meta charset="utf-8"></head>'
        f'<body>{"".join(html_parts)}</body></html>'
    )

    weasyprint.HTML(string=full_html).write_pdf(str(pdf_path))
    print(f'Saved → {pdf_path}')
    return pdf_path


print('profile_and_save() ready.')

In [ ]:
# ── Table selector widget ─────────────────────────────────────────────────────

tables_df  = get_available_tables()
table_opts = [
    (f"{r['table_schema']}.{r['table_name']}  ({r['table_type']}  {int(r['row_count']):,} rows)",
     f"{r['table_schema']}.{r['table_name']}")
    for _, r in tables_df.iterrows()
]

table_dd   = widgets.Dropdown(
    options=table_opts,
    description='Table:',
    layout=widgets.Layout(width='50%')
)
sample_box = widgets.BoundedFloatText(
    value=10, min=0, max=100, step=5,
    description='Sample % (0=all):',
    layout=widgets.Layout(width='22%')
)
run_btn  = widgets.Button(
    description='Profile',
    button_style='primary',
    layout=widgets.Layout(width='12%')
)
save_btn = widgets.Button(
    description='Profile & Save PDF',
    button_style='success',
    layout=widgets.Layout(width='16%')
)
out = widgets.Output()

def on_run(_):
    with out:
        clear_output(wait=True)
        profile_table(table_dd.value, sample_box.value)

def on_save(_):
    with out:
        clear_output(wait=True)
        profile_and_save(table_dd.value, sample_box.value)

run_btn.on_click(on_run)
save_btn.on_click(on_save)

display(widgets.HBox([table_dd, sample_box, run_btn, save_btn]))
display(out)

In [20]:
# ── Profile a table directly (no widget) ─────────────────────────────────────
# sample_pct: Bernoulli % — each row selected independently at random.
# 0 = full table scan. Default is 10%.

# profile_table('DBT_MALEX_STAGING.STG_YELLOW_TRIPS')
# profile_table('DBT_MALEX_STAGING.STG_WEATHER')
# profile_table('DBT_MALEX_STAGING.STG_TAXI_ZONES')
# profile_table('DBT_MALEX_INTERMEDIATE.INT_TRIPS_ENRICHED')
# profile_table('DBT_MALEX_MARTS.FCT_TRIPS')
# profile_table('DBT_MALEX_MARTS.FCT_DAILY_DEMAND')
# profile_table('DBT_MALEX_MARTS.DIM_ZONES')
# profile_table('DBT_MALEX_MARTS.DIM_DATE')
# profile_table('DBT_MALEX.TAXI_ZONE_LOOKUP',  sample_pct=0)   # small table — full scan